## Python Analytics Practice -- NumPy, Pandas, Matplotlib, Seaboarn + EDA & Post-Campaign Analysis

### Domain: Ecommerce - Fashion/Home/Electronics Retail

**What is it in this notebook**
- Every section has **Q1, Q2, Q3 ..** style questions in Markdown, followed by a code cell with the answer.
- All data is **synthetically generated inside this notebook** (Section 0) so it's fully self-contained - no external files needed, safe to publish on GitHub.
- Domain logic (BNPL = buy-now-pay-later, categories = Fashion/Home/Electricals/Beauty, channels = Email/Paid Social/Display/Affiliate/TV) is loosely modelled.

**Sections**

0. Synthetic Data Generation (Customers, Products, Campaigns, Exposures, Orders)
1. Numpy Fundamentals (10 Q&A)
2. Pandas Data Wrangling (15 Q&A)
3. Matplotlib Visualization (10 Q&A)
4. Seaborn Visualization (10 Q&A)
5. In-Depth EDA - "Current Market" Style Analysis (10&A)
6. Campaign / Post-Campaign Analysis - the main focus (15 Q&A)
7. Bonus: Mini Take-Home Style Challenge

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10,15)

RNG = np.random.default_rng(42)
print("Environment ready.")

Environment ready.


## Section 0 - Syntthetic Data Generation

We build 5 linked tables, similar to a star schema.

| Table | Grain | Key fields |
|---|---|---|
| `customers` | 1 row per customer | region, age_band, is_bnpl_user, acquisition_channel |
| `products` | 1 row per SKU | category, sub_category, brand, price, margin_pct |
| `campaigns` | 1 row per campaign | channel, objective, budget, start/end date |
| `exposures` | 1 row per customer-campaign touch | impressions, clicks, exposure_date |
| `orders` | 1 row per order line | revenue, qty, campaign attribution, returned_flag |

The date range covers **12 months** with realistic seasonality spikes around **Black Friday (Nov)** and **Christmas (Dec)**, since that's when a retailer like "`X Group`" sees its biggest campaign activity.

In [62]:
# ----- 0.1 Customers -----
N_CUSTOMERS = 8000

regions = ['London', 'South East', 'North West', 'Midlands', 'Scotland', 'Wales', 'South West', 'North East']
region_weights = [0.18, 0.15, 0.14, 0.16, 0.10, 0.07, 0.10, 0.10]

age_bands = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
age_weights = [0.10, 0.24, 0.23, 0.19, 0.14, 0.10]

acq_channels = ['Paid Social', 'Email', 'Affiliate', 'Organic Search', 'Display', 'TV/Brand']
acq_weights = [0.28, 0.15, 0.12, 0.20, 0.10, 0.15]

signup_start = datetime(2023, 1, 1)
signup_days = RNG.integers(0, 105, N_CUSTOMERS) # up to 3 yrs of tenure

customers = pd.DataFrame({
    'customer_id': [f'CUST{100000+i}' for i in range(N_CUSTOMERS)],
    'signup_date': [signup_start + timedelta(days=int(d)) for d in signup_days],
    'region': RNG.choice(regions, N_CUSTOMERS, p=region_weights),
    'age_band': RNG.choice(age_bands, N_CUSTOMERS, p=age_weights),
    'acquisition_channel': RNG.choice(acq_channels, N_CUSTOMERS, p=acq_weights),
    'is_bnpl_user': RNG.choice([1, 0], N_CUSTOMERS, p=[0.62, 0.38]) # here bnpl penetration is high
})

customers['tenure_days'] = (datetime(2025,12,31) - customers['signup_date']).dt.days

print(customers.shape)
customers.head()

(8000, 7)


,customer_id,signup_date,region,age_band,acquisition_channel,is_bnpl_user,tenure_days
0,CUST100000,2023-01-04,North East,65+,Affiliate,1,1092
1,CUST100001,2023-04-04,Midlands,65+,Paid Social,1,1002
2,CUST100002,2023-01-15,Scotland,25-34,Organic Search,1,1081
3,CUST100003,2023-03-14,Midlands,35-44,TV/Brand,0,1023
4,CUST100004,2023-01-13,North East,35-44,Organic Search,1,1083


In [63]:
# ---- 0.2 Products ----
category_map = {
    'Fashion': ['Womenswear', 'Menswear', 'Kidswear', 'Footwear'],
    'Home': ['Furniture', 'Bedding', 'Kitchenware', 'Decor'],
    'Electricals': ['TV & Audio', 'Small Appliances', 'Large Appliances', 'Tech Accessories'],
    'Beauty': ['Skincare', 'Haircare', 'Fragrance']
}

rows = []
pid = 0
for cat, subs in category_map.items():
    for sub in subs:
        n_skus = RNG.integers(15, 30)
        for _ in range(n_skus):
            base_price = {
                'Fashion': RNG.uniform(12, 120),
                'Home': RNG.uniform(15, 600),
                'Electricals': RNG.uniform(20, 1200),
                'Beauty': RNG.uniform(5,80),
            }[cat]
            rows.append({
                'product_id': f'SKU{pid:05d}',
                'category': cat,
                'sub_category': sub,
                'brand': RNG.choice(['House Brand', 'Premium Partner', 'Budget Line', 'Licensed Brand']),
                'price': round(base_price, 2),
                'margin_pct': round(RNG.uniform(0.18, 0.55), 3),
            })
            pid += 1

products = pd.DataFrame(rows)
print(products.shape)
products.head()

(352, 6)


,product_id,category,sub_category,brand,price,margin_pct
0,SKU00000,Fashion,Womenswear,Budget Line,19.16,0.232
1,SKU00001,Fashion,Womenswear,Premium Partner,118.82,0.252
2,SKU00002,Fashion,Womenswear,Budget Line,23.07,0.368
3,SKU00003,Fashion,Womenswear,Premium Partner,12.30,0.229
4,SKU00004,Fashion,Womenswear,Budget Line,75.93,0.546


In [64]:
# ---- 0.3 Campaigns ----
campaigns_defs = [
    ('CMP001',  'Spring Refresh - Home',            'Email',        'Conversion', '2025-03-03', '2025-03-17', 18000),
    ('CMP002',  'Easter Fashion Push',              'Paid Social',  'Conversion', '2025-04-07', '2025-04-21', 42000),
    ('CMP003',  'Summer Electricals Sale',          'Display',      'Conversion', '2025-06-02', '2025-06-16', 35000),
    ('CMP004',  'Back to School',                   'Paid Social',  'Conversion', '2025-08-18', '2025-09-01', 30000),
    ('CMP005',  'Very Pay Awarness',                'TV',           'Awarness',   '2025-09-15', '2025-10-05', 90000),
    ('CMP006',  'Black Friday Blitzz',              'Paid Social',  'Conversion', '2025-11-21', '2025-11-29', 120000),
    ('CMP007',  'Black Friday Email Blast',         'Email',        'Conversion', '2025-11-24', '2025-11-29', 8000),
    ('CMP008',  'Christmas Gift Guide',             'Display',      'Conversion', '2025-12-01', '2025-12-20', 65000),
    ('CMP009',  'Boxing Day Clearance',             'Email',        'Conversion', '2025-12-26', '2025-12-31', 12000),
    ('CMP010',  'New Year New You - Beauty',        'Affiliate',     'Conversion', '2026-01-01', '2026-01-19', 20000),
]
campaigns = pd.DataFrame(campaigns_defs, columns=[
    'campaign_id', 'campaign_name', 'channel', 'objective', 'start_date', 'end_date', 'budget_gbp'
])
campaigns['start_date'] = pd.to_datetime(campaigns['start_date'])
campaigns['end_date'] = pd.to_datetime(campaigns['end_date'])
campaigns['duration_days'] = (campaigns['end_date'] - campaigns['start_date']).dt.days + 1
campaigns

,campaign_id,campaign_name,channel,objective,start_date,end_date,budget_gbp,duration_days
0,CMP001,Spring Refresh - Home,Email,Conversion,2025-03-03,2025-03-17,18000,15
1,CMP002,Easter Fashion Push,Paid Social,Conversion,2025-04-07,2025-04-21,42000,15
2,CMP003,Summer Electricals Sale,Display,Conversion,2025-06-02,2025-06-16,35000,15
3,CMP004,Back to School,Paid Social,Conversion,2025-08-18,2025-09-01,30000,15
4,CMP005,Very Pay Awarness,TV,Awarness,2025-09-15,2025-10-05,90000,21
5,CMP006,Black Friday Blitzz,Paid Social,Conversion,2025-11-21,2025-11-29,120000,9
6,CMP007,Black Friday Email Blast,Email,Conversion,2025-11-24,2025-11-29,8000,6
7,CMP008,Christmas Gift Guide,Display,Conversion,2025-12-01,2025-12-20,65000,20
8,CMP009,Boxing Day Clearance,Email,Conversion,2025-12-26,2025-12-31,12000,6
9,CMP010,New Year New You - Beauty,Affiliate,Conversion,2026-01-01,2026-01-19,20000,19


In [65]:
# ---- 0.4 Campaign Exposure (impressions/clicks per customer per campaign) ----
exposure_rows = []
for _, c in campaigns.iterrows():
    # not every customer is exposed - sample a target audience size depending on channel reach
    reach_frac = {'TV': 0.55, 'Display': 0.40, 'Paid Social': 0.35, 'Email': 0.30, 'Affiliate': 0.15} [c['channel']]
    audience = customers.sample(frac=reach_frac, random_state=RNG.integers(0, 99999))['customer_id'].values
    n = len(audience)
    exp_dates = c['start_date'] + pd.to_timedelta(RNG.integers(0, c['duration_days'], n), unit='D')
    impressions = RNG.poisson(lam={'TV': 6, 'Display': 8, 'Paid Social': 5, 'Email': 2, 'Affiliate': 3}[c['channel']], size=n)+1
    # base CTR varies by channel, with noise
    base_ctr = {'TV': 0.02, 'Display': 0.012, 'Paid Social': 0.028, 'Email': 0.045, 'Affiliate': 0.020}[c['channel']]
    clicks = RNG.binomial(impressions, np.clip(base_ctr + RNG.normal(0, 0.005, n), 0.0005, None))
    exposure_rows.append(pd.DataFrame({
        'customer_id': audience,
        'campaign_id': c['campaign_id'],
        'exposure_date': exp_dates,
        'impressions': impressions,
        'clicks': clicks
    }))

exposures = pd.concat(exposure_rows, ignore_index=True)
print(exposures.shape)
exposures.head()

(27600, 5)


,customer_id,campaign_id,exposure_date,impressions,clicks
0,CUST104873,CMP001,2025-03-12,3,0
1,CUST100266,CMP001,2025-03-03,3,0
2,CUST106322,CMP001,2025-03-12,2,0
3,CUST101264,CMP001,2025-03-09,7,2
4,CUST100487,CMP001,2025-03-09,1,0


In [73]:
# ---- 0.5 Orders (12 months, with seasonality + campaign-attributed uplift) ----
date_range = pd.date_range('2025-01-01', '2026-01-31', freq='D')

# baseline daily demand multiplier (weekday effect + monthly seasonality incl. Black Friday/Xmas spike)
def seasonality(d):
    month_factor = {1: 1.1, 2: 0.85, 3: 0.95, 4: 1.0, 5: 0.95, 6: 1.0, 7: 0.9, 8: 1.0,
                    9: 1.0, 10: 1.05, 11: 1.6, 12: 1.9}[d.month]
    bf_boost = 2.4 if (d.month == 11 and 21 <= d.day <= 29) else 1.0
    xmas_boost = 1.6 if (d.month == 12 and 1 <= d.day <= 20) else 1.0
    weekend_boost = 1.15 if d.weekday() >= 5 else 1.0
    return month_factor * bf_boost * xmas_boost * weekend_boost

order_rows = []
oid = 0
clicked_pairs = exposures[exposures['clicks']>0][['customer_id', 'campaign_id', 'exposure_date']]
clicked_set = clicked_pairs.groupby(['customer_id']).apply(lambda g: list(zip(g['campaign_id'], g['exposure_date'])), include_groups=False).to_dict()

cust_ids = customers['customer_id'].values
prod_sample = products.sample(2500, replace=True, random_state=1).reset_index(drop=True)

for d in date_range:
    n_orders_today = int(RNG.poisson(70 * seasonality(d)))
    sampled_custs = RNG.choice(cust_ids, n_orders_today)
    sampled_prods = products.sample(n_orders_today, replace=True, weights=None, random_state=RNG.integers(0,99999))
    for cust, (_, prod) in zip(sampled_custs, sampled_prods.iterrows()):
        qty = RNG.choice([1,1,1,2,2,3], 1)[0]
        # check if this customer cicked a campaign within +/- 5 days of order -> attribute ordder to campaign
        attributed_campaign = None
        if cust in clicked_set:
            for camp_id, exp_date in clicked_set[cust]:
                if abs((d - exp_date).days) <=5:
                    attributed_campaign = camp_id
                    break
        returned = RNG.choice([1,0], p=[0.12, 0.88])
        order_rows.append((
            f'ORD{oid:07d}', cust, d, prod['product_id'], int(qty),
            round(prod['price'] * qty, 2), attributed_campaign,
            RNG.choice(['Very Pay (BNPL)', 'Debit/Credit Card', 'PayPal'], p=[0.55, 0.35, 0.10]),
            returned
        ))
        oid += 1

orders = pd.DataFrame(order_rows, columns=[
    'order_id', 'customer_id', 'order_date', 'product_id', 'qty', 'revenue',
    'campaign_id', 'payment_method', 'returned_flag'
])
print(orders.shape)
orders.head()

(35423, 9)


,order_id,customer_id,order_date,product_id,qty,revenue,campaign_id,payment_method,returned_flag
0,ORD0000000,CUST102130,2025-01-01,SKU00063,1,49.48,None,Very Pay (BNPL),0
1,ORD0000001,CUST106663,2025-01-01,SKU00069,1,92.44,None,Debit/Credit Card,0
2,ORD0000002,CUST106074,2025-01-01,SKU00244,2,987.82,None,Very Pay (BNPL),0
3,ORD0000003,CUST103020,2025-01-01,SKU00106,2,177.04,None,Very Pay (BNPL),0
4,ORD0000004,CUST105640,2025-01-01,SKU00259,2,2224.28,None,Very Pay (BNPL),0


In [86]:
# Quick sanity checks before we start practicing
print('Date range: ', orders['order_date'].min(), 'to', orders['order_date'].max())
print('Total revenue (GBP): {:,.0f}'.format(orders['revenue'].sum()))
print('Orders attributed to a campaign: {:,} ({:.1%})'.format(
    orders['campaign_id'].notna().sum(), orders['campaign_id'].notna().mean()))
print("Missing value check:")
print(orders.isna().sum())

Date range:  2025-01-01 00:00:00 to 2026-01-31 00:00:00
Total revenue (GBP): 18,937,722
Orders attributed to a campaign: 629 (1.8%)
Missing value check:
order_id              0
customer_id           0
order_date            0
product_id            0
qty                   0
revenue               0
campaign_id       34794
payment_method        0
returned_flag         0
dtype: int64


**Note on realism:** This synthetic data - the actual click-through rates, attribution window, and conversion uplift here are simplified for practice purposes. In a ecommerce / analytics learning, you'd reference your own observed metrics (eg: attribution windows, MMM vs last-touch databases) rather than the exact numbers below. Use this dataset to **practice the mechanics** of the analysis, not to memorize these numbers as facts.